<h1>1. Import library</h1>

In [1]:
############# Importing System Libraries #############
import sys
import os
import cv2
project_dir = "/data/atran16/ProteinClassification_3D"

############# Importing support Libraries #############
import json
import random
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import FormatStrFormatter
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
############# Importing datasets classes #############

sys.path.insert(0, f"{project_dir}/utils/datasets")
from pdb_ds import test_tf, get_classes, real_protein_testset

sys.path.insert(0, f"{project_dir}/evaluations")
from evaluation_pdb import realTest_cm
############# Importing models #############
sys.path.insert(0, f"{project_dir}")
from models import (
    load_Resnet,
    load_ConvNeXt,
    load_CoAtNet,
    load_EfficientNetV2,
    load_VIT_SizeT,
    load_RegNetY16GF,
    load_SwinV2B,
)


class_names = get_classes("/data/atran16/ProteinClassification_3D/3D_PDB_5013/PNG126")
topk=(1,3,5,10,20, 50, 100, 200, 500)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
configs = {"model": "Resnet152", 
           "image_size": 224,
           "n_classes":len(class_names), 
           "test_image_path": "/data/atran16/ProteinClassification_3D/3D_PDB_Dataset/testingDataFromProfessorSu",
           "pretrained_path":"/data/atran16/ProteinClassification_3D/trained_results/04012026_train_126_30/Resnet152/PDBRSTuan.pt",
           "full_rs_dir": "/data/atran16/ProteinClassification_3D/trained_results/04012026_train_126_30/Resnet152"
           }
configs['rs_dir'] = os.path.join(configs['full_rs_dir'], "PDBRSTuan.pt")
configs["image_size"] = (configs["image_size"], configs["image_size"])

<h1>2. Loading models</h1>

In [2]:
if "Resnet" in configs["model"]:
    model = load_Resnet(name = configs["model"], num_classes = configs["n_classes"], pretrained_path=configs["pretrained_path"], device=device)
    model = model.to(device)
elif configs["model"] == "ConvNeXt":
    model = load_ConvNeXt(num_classes = configs["n_classes"], pretrained_path=configs["pretrained_path"], device=device)
    model = model.to(device)
    print("Loading ConvNeXt model successfully!\n")
elif "CoAtNet" in configs["model"]:
    model = load_CoAtNet(name = configs["model"], num_classes = configs["n_classes"], pretrained_path=configs["pretrained_path"], device=device)
    model = model.to(device)
    print("Loading CoAtNet2 model successfully!\n")
elif "EfficientNetV2" in configs["model"]:
    model = load_EfficientNetV2(name = configs["model"], num_classes = configs["n_classes"], pretrained_path=configs["pretrained_path"], device=device)
    model = model.to(device)
elif configs["model"] == "MaxViT":
    model = load_VIT_SizeT(num_classes = configs["n_classes"], pretrained_path=configs["pretrained_path"], device=device)
    model = model.to(device)
    print("Loading MaxViT_SizeT model successfully!\n")
elif configs["model"] == "RegNetY16GF":
    model = load_RegNetY16GF(num_classes = configs["n_classes"], pretrained_path=configs["pretrained_path"], device=device)
    model = model.to(device)
    print("Loading RegNetY16GF model successfully!\n")
elif configs["model"] == "SwinV2B":
    model = load_SwinV2B(num_classes = configs["n_classes"], pretrained_path=configs["pretrained_path"], device=device)
    model = model.to(device)
    print("Loading SwinV2B model successfully!\n")
else:
    raise ValueError(f"Unsupported model type: {configs['model']}")

Loading ResNet152 model successfully!

Pretrained '/data/atran16/ProteinClassification_3D/trained_results/04012026_train_126_30/Resnet152/PDBRSTuan.pt' model loaded successfully!


<h1>3. Confusion Matrix</h1>

In [3]:
images_per_class, labels_per_class = real_protein_testset(configs["test_image_path"], class_names)
for k in topk:
    realTest_cm(
        image_size=configs["image_size"],
        class_names=class_names,
        checkpoint_path=configs["rs_dir"],
        device=device,
        model=model,
        path2save=configs["full_rs_dir"],
        images_per_class=images_per_class,
        labels_per_class=labels_per_class,
        top_k = k,
        saveStatisticsReport=True
    )

Top-1 acceptance accuracy on real dataset:0/34(0.00%)
Classification report saved to: /data/atran16/ProteinClassification_3D/trained_results/04012026_train_126_30/Resnet152/classification_report.csv
Top-3 acceptance accuracy on real dataset:1/34(2.94%)
Classification report saved to: /data/atran16/ProteinClassification_3D/trained_results/04012026_train_126_30/Resnet152/classification_report.csv
Top-5 acceptance accuracy on real dataset:1/34(2.94%)
Classification report saved to: /data/atran16/ProteinClassification_3D/trained_results/04012026_train_126_30/Resnet152/classification_report.csv
Top-10 acceptance accuracy on real dataset:1/34(2.94%)
Classification report saved to: /data/atran16/ProteinClassification_3D/trained_results/04012026_train_126_30/Resnet152/classification_report.csv
Top-20 acceptance accuracy on real dataset:1/34(2.94%)
Classification report saved to: /data/atran16/ProteinClassification_3D/trained_results/04012026_train_126_30/Resnet152/classification_report.csv
Top